# Module 2: Query Optimization for Better Retrieval

## Learning Objectives

By completing this notebook, you will:
1. Understand why basic semantic search sometimes fails
2. Implement query rewriting techniques to improve retrieval
3. Use HyDE (Hypothetical Document Embeddings) for better matching
4. Build self-query retrieval systems with metadata filters
5. Measure and optimize retrieval quality metrics

## Prerequisites

- Completed Module 2, Notebooks 1-2 (RAG and Research Agent)
- Understanding of vector embeddings
- Knowledge of semantic search basics
- LLM API access

## Estimated Time: 80-100 minutes

---

## 1. The Query-Document Mismatch Problem

**Problem:** Queries and documents use different language.

**Example:**
- **Query:** "Why are oil prices going up?"
- **Document:** "Crude oil inventories declined by 5.2 million barrels... production cuts by OPEC+..."

The query asks "why" (causal language), but documents describe facts (inventory data). Semantic search might miss the connection.

**Solutions:**
1. **Query Rewriting**: Expand "why prices up" → "inventory declines, production cuts, demand increase"
2. **HyDE**: Generate hypothetical answer, search for documents similar to that
3. **Self-Query**: Extract metadata filters from query (dates, categories)
4. **Multi-Query**: Generate multiple query variations, combine results

In [ ]:
# Purpose: Setup imports and load vector database
# Key Concept: Query optimization happens before vector search

import os
import json
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Any, Tuple
from pathlib import Path

import pandas as pd
import numpy as np
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import seaborn as sns

# LLM imports
import openai
from anthropic import Anthropic

# Vector database
import chromadb
from sentence_transformers import SentenceTransformer

# Configuration
load_dotenv()
sns.set_style('whitegrid')

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')

# Initialize clients
openai_client = openai.OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

# Data directories
CHROMA_DIR = Path('data/chroma_db')

# Load vector database
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
try:
    collection = chroma_client.get_collection(name="eia_petroleum_reports")
    print(f"✅ Loaded EIA vector database ({collection.count()} documents)")
except:
    collection = None
    print("⚠️  No vector database found")

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("✅ Setup complete")
print(f"   LLM: {'OpenAI' if openai_client else 'Claude' if anthropic_client else 'None'}")

## 2. Query Rewriting with LLMs

**Technique:** Use LLM to rewrite user's query into better search terms.

**Rewriting Strategies:**
1. **Keyword Expansion**: Add synonyms and related terms
2. **Specificity**: Make vague queries more specific
3. **Domain Translation**: Convert casual language to technical terms
4. **Multi-Angle**: Generate multiple query perspectives

**Example:**
- Original: "oil shortage"
- Rewritten: "crude oil inventory decline supply disruption production decrease"

In [ ]:
# Purpose: Implement query rewriting with LLM
# Key Concept: Better queries = better retrieval results

def rewrite_query(query: str, strategy: str = 'expand') -> str:
    """
    Rewrite user query for better retrieval.
    
    Parameters:
    -----------
    query : str
        Original user query
    strategy : str
        Rewriting strategy: 'expand', 'specific', 'technical', 'multi'
        
    Returns:
    --------
    str
        Rewritten query
    """
    if strategy == 'expand':
        prompt = f"""
Rewrite this query by expanding it with synonyms and related terms for better semantic search.
Focus on commodity market terminology.

Original query: {query}

Rewritten query:
"""
    
    elif strategy == 'specific':
        prompt = f"""
Make this query more specific and concrete for searching petroleum market reports.
Add specific metrics, products, and market indicators.

Original query: {query}

More specific query:
"""
    
    elif strategy == 'technical':
        prompt = f"""
Translate this casual query into technical petroleum industry language.
Use official EIA terminology.

Original query: {query}

Technical query:
"""
    
    elif strategy == 'multi':
        prompt = f"""
Generate 3 different ways to ask this question, each emphasizing a different aspect
relevant to petroleum market analysis. Return as JSON array.

Original query: {query}

JSON array of 3 rewritten queries:
"""
    else:
        return query
    
    try:
        if openai_client:
            response = openai_client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
                max_tokens=200
            )
            rewritten = response.choices[0].message.content.strip()
        
        elif anthropic_client:
            message = anthropic_client.messages.create(
                model="claude-3-5-sonnet-20241022",
                max_tokens=200,
                temperature=0.3,
                messages=[{"role": "user", "content": prompt}]
            )
            rewritten = message.content[0].text.strip()
        
        else:
            rewritten = query
        
        return rewritten
    
    except Exception as e:
        print(f"Error rewriting query: {e}")
        return query

# Test query rewriting
test_queries = [
    "oil shortage",
    "why are gas prices high",
    "refinery problems"
]

print("Query Rewriting Examples\n")
print("="*70)

for query in test_queries:
    print(f"\nOriginal: {query}")
    print("-"*70)
    
    for strategy in ['expand', 'specific', 'technical']:
        rewritten = rewrite_query(query, strategy=strategy)
        print(f"  {strategy.capitalize()}: {rewritten}")

print("\n" + "="*70)

### 💡 Exercise 2.1: Compare Retrieval Quality

**Task:** Compare search results using original vs. rewritten queries.

**Requirements:**
1. Choose a vague or casual query
2. Search using original query
3. Rewrite query using different strategies
4. Search using each rewritten version
5. Compare result quality:
   - Relevance (are results on-topic?)
   - Specificity (do they answer the question?)
   - Coverage (do they cover different aspects?)

**Expected Output:** Analysis showing which rewriting strategy works best

In [ ]:
# YOUR CODE HERE
# ---------------





In [ ]:
# SOLUTION
# --------

def search_with_query(query: str, top_k: int = 3) -> List[Dict]:
    """Search vector database with query."""
    if not collection:
        return []
    
    query_embedding = embedding_model.encode([query])[0]
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )
    
    formatted = []
    if results['ids'] and len(results['ids'][0]) > 0:
        for i in range(len(results['ids'][0])):
            formatted.append({
                'text': results['documents'][0][i],
                'metadata': results['metadatas'][0][i],
                'distance': results['distances'][0][i] if 'distances' in results else None
            })
    
    return formatted

if collection:
    test_query = "why are gasoline prices high"
    
    print("Comparing Retrieval Quality\n")
    print("="*70)
    print(f"Original Query: '{test_query}'\n")
    
    # Original query
    print("[1] Original Query Results")
    print("-"*70)
    original_results = search_with_query(test_query, top_k=2)
    for i, result in enumerate(original_results, 1):
        print(f"Result {i}: {result['text'][:150]}...")
        if result['distance']:
            print(f"  Similarity: {1 - result['distance']:.3f}")
    
    # Rewritten queries
    strategies = ['expand', 'specific', 'technical']
    
    for strategy in strategies:
        rewritten = rewrite_query(test_query, strategy=strategy)
        print(f"\n[2] {strategy.capitalize()} Strategy")
        print("-"*70)
        print(f"Rewritten: '{rewritten}'")
        
        results = search_with_query(rewritten, top_k=2)
        for i, result in enumerate(results, 1):
            print(f"Result {i}: {result['text'][:150]}...")
            if result['distance']:
                print(f"  Similarity: {1 - result['distance']:.3f}")
    
    print("\n" + "="*70)
    print("Analysis:")
    print("  • Technical queries typically retrieve more specific data")
    print("  • Expanded queries cast wider net but may have lower precision")
    print("  • Specific queries balance relevance and coverage")
else:
    print("⚠️  Cannot test without vector database")

## 3. HyDE: Hypothetical Document Embeddings

**Innovative Technique:** Instead of searching with the query, search with a hypothetical answer.

**How HyDE Works:**
1. User asks: "Why are oil prices rising?"
2. LLM generates hypothetical answer: "Oil prices are rising due to OPEC production cuts, declining inventories, and strong demand..."
3. Embed the hypothetical answer
4. Search for documents similar to hypothetical answer
5. Use actual documents to generate real answer

**Why It Works:**
- Hypothetical answers use document-like language
- Better semantic match than queries
- Especially good for "why" and "how" questions

In [ ]:
# Purpose: Implement HyDE for improved retrieval
# Key Concept: Search with hypothetical answer, not the question

def hyde_search(query: str, top_k: int = 5) -> Dict[str, Any]:
    """
    Perform HyDE (Hypothetical Document Embeddings) search.
    
    Parameters:
    -----------
    query : str
        User's question
    top_k : int
        Number of results
        
    Returns:
    --------
    dict
        Hypothetical answer, search results, and metadata
    """
    # Step 1: Generate hypothetical answer
    prompt = f"""
You are writing an excerpt from an EIA petroleum market report.
Write a factual paragraph that would answer this question, using typical report language
with specific metrics, dates, and market terminology.

Question: {query}

Report excerpt:
"""
    
    try:
        if openai_client:
            response = openai_client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.5,
                max_tokens=200
            )
            hypothetical_answer = response.choices[0].message.content.strip()
        
        elif anthropic_client:
            message = anthropic_client.messages.create(
                model="claude-3-5-sonnet-20241022",
                max_tokens=200,
                temperature=0.5,
                messages=[{"role": "user", "content": prompt}]
            )
            hypothetical_answer = message.content[0].text.strip()
        
        else:
            return {'error': 'No LLM configured'}
        
    except Exception as e:
        return {'error': f'Failed to generate hypothetical answer: {str(e)}'}
    
    # Step 2: Embed hypothetical answer
    if not collection:
        return {'error': 'No vector database'}
    
    hypothesis_embedding = embedding_model.encode([hypothetical_answer])[0]
    
    # Step 3: Search with hypothesis embedding
    results = collection.query(
        query_embeddings=[hypothesis_embedding.tolist()],
        n_results=top_k
    )
    
    # Step 4: Format results
    retrieved_docs = []
    if results['ids'] and len(results['ids'][0]) > 0:
        for i in range(len(results['ids'][0])):
            retrieved_docs.append({
                'text': results['documents'][0][i],
                'metadata': results['metadatas'][0][i],
                'distance': results['distances'][0][i] if 'distances' in results else None
            })
    
    return {
        'hypothetical_answer': hypothetical_answer,
        'retrieved_documents': retrieved_docs,
        'num_results': len(retrieved_docs)
    }

# Test HyDE
if collection and (openai_client or anthropic_client):
    test_query = "Why have crude oil inventories been changing recently?"
    
    print("HyDE Search Example\n")
    print("="*70)
    print(f"Query: {test_query}\n")
    
    result = hyde_search(test_query, top_k=3)
    
    if 'error' not in result:
        print("[Step 1] Hypothetical Answer Generated:")
        print("-"*70)
        print(result['hypothetical_answer'])
        
        print("\n[Step 2] Retrieved Actual Documents:")
        print("-"*70)
        for i, doc in enumerate(result['retrieved_documents'], 1):
            print(f"\nDocument {i}:")
            print(f"  Date: {doc['metadata'].get('report_date', 'unknown')}")
            print(f"  Text: {doc['text'][:200]}...")
            if doc['distance']:
                print(f"  Similarity to hypothesis: {1 - doc['distance']:.3f}")
    else:
        print(f"Error: {result['error']}")
else:
    print("⚠️  HyDE requires vector database and LLM")

### 💡 Exercise 3.1: HyDE vs Standard Search

**Task:** Compare HyDE against standard semantic search for complex questions.

**Test Questions** (choose 2-3):
1. "What factors are driving changes in refinery operations?"
2. "How do seasonal patterns affect gasoline demand?"
3. "Why would declining production impact inventory levels?"

**Requirements:**
- Run standard search (direct query embedding)
- Run HyDE search (hypothesis embedding)
- Compare:
  - Relevance of top results
  - Similarity scores
  - Coverage of the question
- Determine which approach works better for each question type

**Hint:** HyDE typically excels at "why" and "how" questions

In [ ]:
# YOUR CODE HERE
# ---------------





In [ ]:
# SOLUTION
# --------

if collection and (openai_client or anthropic_client):
    test_questions = [
        "What factors are driving changes in refinery operations?",
        "How do seasonal patterns affect gasoline demand?"
    ]
    
    print("HyDE vs Standard Search Comparison\n")
    print("="*70)
    
    for question in test_questions:
        print(f"\nQuestion: {question}")
        print("="*70)
        
        # Standard search
        print("\n[A] Standard Semantic Search")
        print("-"*70)
        standard_results = search_with_query(question, top_k=2)
        for i, result in enumerate(standard_results, 1):
            print(f"Result {i}: {result['text'][:120]}...")
            if result['distance']:
                print(f"  Similarity: {1 - result['distance']:.3f}")
        
        # HyDE search
        print("\n[B] HyDE Search")
        print("-"*70)
        hyde_result = hyde_search(question, top_k=2)
        
        if 'error' not in hyde_result:
            print(f"Hypothesis: {hyde_result['hypothetical_answer'][:120]}...\n")
            for i, doc in enumerate(hyde_result['retrieved_documents'], 1):
                print(f"Result {i}: {doc['text'][:120]}...")
                if doc['distance']:
                    print(f"  Similarity: {1 - doc['distance']:.3f}")
        
        print("\n" + "-"*70)
    
    print("\n" + "="*70)
    print("Key Findings:")
    print("  • HyDE often retrieves more contextually relevant documents")
    print("  • Standard search may have higher similarity scores but less relevance")
    print("  • HyDE works best for analytical/causal questions")
    print("  • Standard search works well for factual/data queries")
else:
    print("⚠️  Comparison requires vector database and LLM")

## 4. Self-Query Retrieval with Metadata

**Problem:** Users often ask questions with implicit filters.

**Example:** "What were crude oil inventories in Q1 2024?"
- Semantic part: "crude oil inventories"
- Metadata filter: date range (2024-01-01 to 2024-03-31)

**Self-Query Approach:**
1. LLM extracts metadata filters from query
2. Separate semantic search query from filters
3. Apply both to vector database
4. Get results matching semantic meaning AND metadata

In [ ]:
# Purpose: Implement self-query retrieval
# Key Concept: LLM extracts structured filters from natural language

def extract_query_components(query: str) -> Dict[str, Any]:
    """
    Extract semantic query and metadata filters from natural language query.
    
    Parameters:
    -----------
    query : str
        Natural language query
        
    Returns:
    --------
    dict
        semantic_query, metadata_filters, and reasoning
    """
    prompt = f"""
Analyze this query and extract:
1. The semantic search query (what to search for)
2. Any metadata filters (date ranges, categories, etc.)

Query: {query}

Return JSON:
{{
  "semantic_query": "the core question for semantic search",
  "metadata_filters": {{
    "date_start": "YYYY-MM-DD or null",
    "date_end": "YYYY-MM-DD or null",
    "category": "product type or null"
  }},
  "reasoning": "brief explanation"
}}
"""
    
    try:
        if openai_client:
            response = openai_client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                response_format={"type": "json_object"}
            )
            result = json.loads(response.choices[0].message.content)
        
        elif anthropic_client:
            message = anthropic_client.messages.create(
                model="claude-3-5-sonnet-20241022",
                max_tokens=500,
                temperature=0.0,
                messages=[{"role": "user", "content": prompt}]
            )
            response_text = message.content[0].text
            if "```json" in response_text:
                response_text = response_text.split("```json")[1].split("```")[0]
            result = json.loads(response_text.strip())
        
        else:
            result = {'error': 'No LLM configured'}
        
        return result
    
    except Exception as e:
        return {'error': str(e)}

def self_query_search(query: str, top_k: int = 5) -> Dict[str, Any]:
    """
    Perform self-query retrieval with metadata filtering.
    
    Parameters:
    -----------
    query : str
        Natural language query with potential filters
    top_k : int
        Number of results
        
    Returns:
    --------
    dict
        Extracted components, search results, and metadata
    """
    # Extract components
    components = extract_query_components(query)
    
    if 'error' in components:
        return {'error': components['error']}
    
    if not collection:
        return {'error': 'No vector database'}
    
    # Build semantic query
    semantic_query = components.get('semantic_query', query)
    query_embedding = embedding_model.encode([semantic_query])[0]
    
    # Build metadata filter
    metadata_filters = components.get('metadata_filters', {})
    where_filter = None
    
    if metadata_filters:
        where_filter = {}
        if metadata_filters.get('date_start'):
            where_filter['report_date'] = {'$gte': metadata_filters['date_start']}
        if metadata_filters.get('date_end'):
            if 'report_date' not in where_filter:
                where_filter['report_date'] = {}
            where_filter['report_date']['$lte'] = metadata_filters['date_end']
    
    # Search with filters
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k,
        where=where_filter if where_filter else None
    )
    
    # Format results
    retrieved_docs = []
    if results['ids'] and len(results['ids'][0]) > 0:
        for i in range(len(results['ids'][0])):
            retrieved_docs.append({
                'text': results['documents'][0][i],
                'metadata': results['metadatas'][0][i],
                'distance': results['distances'][0][i] if 'distances' in results else None
            })
    
    return {
        'original_query': query,
        'semantic_query': semantic_query,
        'metadata_filters': metadata_filters,
        'reasoning': components.get('reasoning', ''),
        'retrieved_documents': retrieved_docs,
        'num_results': len(retrieved_docs)
    }

# Test self-query
if collection and (openai_client or anthropic_client):
    test_query = "What were crude oil inventories in early 2024?"
    
    print("Self-Query Retrieval Example\n")
    print("="*70)
    print(f"Query: {test_query}\n")
    
    result = self_query_search(test_query, top_k=3)
    
    if 'error' not in result:
        print("[Query Decomposition]")
        print("-"*70)
        print(f"Semantic Query: {result['semantic_query']}")
        print(f"Metadata Filters: {json.dumps(result['metadata_filters'], indent=2)}")
        print(f"Reasoning: {result['reasoning']}")
        
        print("\n[Retrieved Documents]")
        print("-"*70)
        for i, doc in enumerate(result['retrieved_documents'], 1):
            print(f"\nDocument {i}:")
            print(f"  Date: {doc['metadata'].get('report_date', 'unknown')}")
            print(f"  Text: {doc['text'][:150]}...")
    else:
        print(f"Error: {result['error']}")
else:
    print("⚠️  Self-query requires vector database and LLM")

### 💡 Exercise 4.1: Multi-Query Fusion

**Task:** Implement multi-query retrieval that combines results from multiple query variations.

**Approach:**
1. Generate 3-5 variations of the original query
2. Search with each variation
3. Combine results using Reciprocal Rank Fusion (RRF)
4. Return top K merged results

**RRF Formula:**
```
score(doc) = sum over queries: 1 / (k + rank_in_query)
where k = 60 (constant)
```

**Requirements:**
- Create function `multi_query_search(query, num_variations=3, top_k=5)`
- Generate query variations using LLM
- Search with each variation
- Implement RRF to merge results
- Return ranked, deduplicated results

**Expected Output:** More robust results that don't depend on single query phrasing

In [ ]:
# YOUR CODE HERE
# ---------------





In [ ]:
# SOLUTION
# --------

def multi_query_search(query: str, num_variations: int = 3, top_k: int = 5) -> Dict[str, Any]:
    """
    Multi-query retrieval with Reciprocal Rank Fusion.
    
    Parameters:
    -----------
    query : str
        Original query
    num_variations : int
        Number of query variations to generate
    top_k : int
        Final number of results
        
    Returns:
    --------
    dict
        Query variations, merged results, and scores
    """
    if not collection or not (openai_client or anthropic_client):
        return {'error': 'Requires vector database and LLM'}
    
    # Generate query variations
    prompt = f"""
Generate {num_variations} different ways to phrase this search query.
Each variation should emphasize a different aspect but seek the same information.
Return as JSON array of strings.

Original query: {query}

JSON array:
"""
    
    try:
        if openai_client:
            response = openai_client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                response_format={"type": "json_object"}
            )
            result = json.loads(response.choices[0].message.content)
            variations = result.get('queries', result.get('variations', [query]))
        elif anthropic_client:
            message = anthropic_client.messages.create(
                model="claude-3-5-sonnet-20241022",
                max_tokens=300,
                temperature=0.7,
                messages=[{"role": "user", "content": prompt}]
            )
            response_text = message.content[0].text
            if "```json" in response_text:
                response_text = response_text.split("```json")[1].split("```")[0]
            result = json.loads(response_text.strip())
            variations = result.get('queries', result.get('variations', [query]))
        else:
            variations = [query]
    except:
        variations = [query]
    
    # Ensure we have list
    if isinstance(variations, dict):
        variations = list(variations.values())
    
    # Add original query
    all_queries = [query] + variations[:num_variations]
    
    # Search with each query
    all_results = {}
    doc_scores = {}  # For RRF
    
    for q in all_queries:
        results = search_with_query(q, top_k=top_k * 2)
        all_results[q] = results
        
        # Calculate RRF scores
        for rank, doc in enumerate(results):
            doc_id = doc['text'][:100]  # Use text snippet as ID
            rrf_score = 1.0 / (60 + rank + 1)
            if doc_id in doc_scores:
                doc_scores[doc_id]['score'] += rrf_score
            else:
                doc_scores[doc_id] = {
                    'score': rrf_score,
                    'doc': doc
                }
    
    # Sort by RRF score
    sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1]['score'], reverse=True)
    
    # Get top K
    final_results = [
        {
            **doc_data['doc'],
            'rrf_score': doc_data['score']
        }
        for doc_id, doc_data in sorted_docs[:top_k]
    ]
    
    return {
        'original_query': query,
        'query_variations': all_queries,
        'merged_results': final_results,
        'num_results': len(final_results)
    }

# Test multi-query search
if collection and (openai_client or anthropic_client):
    test_query = "refinery utilization rates"
    
    print("Multi-Query Search with RRF\n")
    print("="*70)
    print(f"Original Query: {test_query}\n")
    
    result = multi_query_search(test_query, num_variations=2, top_k=3)
    
    if 'error' not in result:
        print("[Query Variations]")
        print("-"*70)
        for i, var in enumerate(result['query_variations'], 1):
            print(f"{i}. {var}")
        
        print("\n[Merged Results (RRF)]")
        print("-"*70)
        for i, doc in enumerate(result['merged_results'], 1):
            print(f"\nResult {i}:")
            print(f"  RRF Score: {doc['rrf_score']:.4f}")
            print(f"  Date: {doc['metadata'].get('report_date', 'unknown')}")
            print(f"  Text: {doc['text'][:120]}...")
    else:
        print(f"Error: {result['error']}")
else:
    print("⚠️  Multi-query requires vector database and LLM")

## Summary

### Key Takeaways

1. **Query-Document Mismatch is Real** - Simple semantic search often fails

2. **Query Rewriting Improves Results** - Expand, specify, or translate queries

3. **HyDE Works for Causal Questions** - Search with hypothetical answers

4. **Self-Query Extracts Filters** - Separate semantics from metadata

5. **Multi-Query Increases Robustness** - RRF merges results from variations

### Skills Gained

✅ Implementing query rewriting strategies  
✅ Using HyDE for improved retrieval  
✅ Building self-query systems with metadata  
✅ Multi-query fusion with RRF  
✅ Comparing retrieval quality metrics  
✅ Optimizing RAG pipelines  

### What's Next

In **Module 3, Notebook 2** (`02_sentiment_aggregation.ipynb`), we'll learn:
- Aggregating sentiment across multiple sources
- Weighting schemes for different sources
- Time decay functions for sentiment
- Building composite sentiment indicators

### Additional Resources

- **HyDE Paper:** https://arxiv.org/abs/2212.10496
- **RRF Algorithm:** Reciprocal Rank Fusion for multi-query retrieval
- **Query Optimization:** LangChain documentation on retrieval strategies

---

**Next:** [Module 3, Notebook 2 - Sentiment Aggregation](../../module_03_sentiment/notebooks/02_sentiment_aggregation.ipynb)